# CellAgent Agent Reasoning

Run post-SCA reasoning and scoring from the pipeline manifest and a multimodal prior JSON/JSONL.


In [1]:
from pathlib import Path
import json
import subprocess
import sys
import yaml

PROJECT_ROOT = Path('/root/wanghaoran/zxy/project/cellagent')
CONFIG = PROJECT_ROOT / 'config/config.yaml'
RAG_CONFIG = PROJECT_ROOT / 'config/rag_sources.yaml'
cfg = yaml.safe_load(CONFIG.read_text())
OUTPUT_ROOT = Path(cfg['pipeline']['output_root'])
MANIFEST = OUTPUT_ROOT / 'pipeline_manifest.json'
MULTIMODAL_PRIOR = OUTPUT_ROOT / 'multimodal_prior' / 'cluster_predictions.jsonl'

print(json.dumps({
    'python': sys.executable,
    'config': str(CONFIG),
    'manifest': str(MANIFEST),
    'default_prior_json': str(MULTIMODAL_PRIOR),
}, ensure_ascii=False, indent=2))


{
  "python": "/root/miniconda3/envs/cellagent/bin/python3.10",
  "config": "/root/wanghaoran/zxy/project/cellagent/config/config.yaml",
  "manifest": "/home/qijinyin/wanghaoran/zxy/cellagent_output/pipeline_manifest.json",
  "default_prior_json": "/home/qijinyin/wanghaoran/zxy/cellagent_output/multimodal_prior/cluster_predictions.jsonl"
}


In [2]:
def require_exists(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    return path

def run_cmd(cmd):
    full_cmd = [sys.executable] + [str(x) for x in cmd]
    print('[run]', ' '.join(full_cmd))
    subprocess.run(full_cmd, cwd=PROJECT_ROOT, check=True)

require_exists(MANIFEST)
require_exists(MULTIMODAL_PRIOR)


PosixPath('/home/qijinyin/wanghaoran/zxy/cellagent_output/multimodal_prior/cluster_predictions.jsonl')

In [3]:
# Run deterministic reasoning + scoring + final aggregation.
run_cmd([
    'scripts/run_agent.py',
    '--config', CONFIG,
    '--rag-config', RAG_CONFIG,
    '--manifest', MANIFEST,
    '--prior-json', MULTIMODAL_PRIOR,
])

summary_path = OUTPUT_ROOT / 'agent_run_summary.json'
require_exists(summary_path)
print(json.dumps(json.loads(summary_path.read_text()), ensure_ascii=False, indent=2))


[run] /root/miniconda3/envs/cellagent/bin/python3.10 scripts/run_agent.py --config /root/wanghaoran/zxy/project/cellagent/config/config.yaml --rag-config /root/wanghaoran/zxy/project/cellagent/config/rag_sources.yaml --manifest /home/qijinyin/wanghaoran/zxy/cellagent_output/pipeline_manifest.json --prior-json /home/qijinyin/wanghaoran/zxy/cellagent_output/multimodal_prior/cluster_predictions.jsonl


{
  "reasoning_paths": [
    "/home/qijinyin/wanghaoran/zxy/cellagent_output/reasoning/case_smoke_astro_c11_cluster_11_reasoning.json"
  ],
  "judge_paths": [
    "/home/qijinyin/wanghaoran/zxy/cellagent_output/judging/case_smoke_astro_c11_cluster_11_judge.json"
  ],
  "final_json": "/home/qijinyin/wanghaoran/zxy/cellagent_output/final/annotations.json",
  "final_csv": "/home/qijinyin/wanghaoran/zxy/cellagent_output/final/annotations.csv"
}


{
  "reasoning_paths": [
    "/home/qijinyin/wanghaoran/zxy/cellagent_output/reasoning/case_smoke_astro_c11_cluster_11_reasoning.json"
  ],
  "judge_paths": [
    "/home/qijinyin/wanghaoran/zxy/cellagent_output/judging/case_smoke_astro_c11_cluster_11_judge.json"
  ],
  "final_json": "/home/qijinyin/wanghaoran/zxy/cellagent_output/final/annotations.json",
  "final_csv": "/home/qijinyin/wanghaoran/zxy/cellagent_output/final/annotations.csv"
}


In [4]:
# Inspect final outputs.
final_json = OUTPUT_ROOT / 'final' / 'annotations.json'
final_csv = OUTPUT_ROOT / 'final' / 'annotations.csv'
if final_json.exists():
    print(json.dumps(json.loads(final_json.read_text()), ensure_ascii=False, indent=2)[:5000])
if final_csv.exists():
    print(final_csv.read_text().splitlines()[:20])


[
  {
    "cluster_id": "11",
    "cell_type": "astrocyte",
    "cell_type_cl_id": "CL:0000127",
    "overall_score": 80,
    "status": "accepted",
    "veto_triggered": false,
    "veto_reason": "",
    "marker_match_score": 40,
    "conflict_penalty": 30,
    "function_consistency_score": 10,
    "tissue_consistency_score": 0,
    "marker_matches": "MT2A,SLC1A3,CLU,APOE,MT1E,S100B,CST3",
    "marker_misses": "FBXL7,PARD3,PITPNC1"
  }
]
['cluster_id,cell_type,cell_type_cl_id,overall_score,status,veto_triggered,veto_reason,marker_match_score,conflict_penalty,function_consistency_score,tissue_consistency_score,marker_matches,marker_misses', '11,astrocyte,CL:0000127,80,accepted,False,,40,30,10,0,"MT2A,SLC1A3,CLU,APOE,MT1E,S100B,CST3","FBXL7,PARD3,PITPNC1"']
